<a href="https://colab.research.google.com/github/emanhassan2020/HandsOn/blob/main/Agents/HuggingFace/web_browser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Installation
! pip install smolagents
# To install from source instead of the last release, comment the command above and uncomment the following one.
# ! pip install git+https://github.com/huggingface/smolagents.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 3.4 MB/s eta 0:00:00


# Web Browser Automation with Agents 🤖🌐

In this notebook, we'll create an **agent-powered web browser automation system**! This system can navigate websites, interact with elements, and extract information automatically.

The agent will be able to:

- [x] Navigate to web pages
- [x] Click on elements
- [x] Search within pages
- [x] Handle popups and modals
- [x] Extract information

Let's set up this system step by step!

First, run these lines to install the required dependencies:

```bash
pip install smolagents selenium helium pillow -q
```

Let's import our required libraries and set up environment variables:

In [2]:
!pip install smolagents selenium helium pillow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 8.5 MB/s eta 0:00:00


In [3]:
from io import BytesIO
from time import sleep

import helium
from dotenv import load_dotenv
from PIL import Image
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

from smolagents import CodeAgent, tool
from smolagents.agents import ActionStep

# Load environment variables
load_dotenv()

False

Now let's create our core browser interaction tools that will allow our agent to navigate and interact with web pages:

In [4]:
@tool
def search_item_ctrl_f(text: str, nth_result: int = 1) -> str:
    """
    Searches for text on the current page via Ctrl + F and jumps to the nth occurrence.
    Args:
        text: The text to search for
        nth_result: Which occurrence to jump to (default: 1)
    """
    elements = driver.find_elements(By.XPATH, f"//*[contains(text(), '{text}')]")
    if nth_result > len(elements):
        raise Exception(f"Match n°{nth_result} not found (only {len(elements)} matches found)")
    result = f"Found {len(elements)} matches for '{text}'."
    elem = elements[nth_result - 1]
    driver.execute_script("arguments[0].scrollIntoView(true);", elem)
    result += f"Focused on element {nth_result} of {len(elements)}"
    return result

@tool
def go_back() -> None:
    """Goes back to previous page."""
    driver.back()

@tool
def close_popups() -> str:
    """
    Closes any visible modal or pop-up on the page. Use this to dismiss pop-up windows!
    This does not work on cookie consent banners.
    """
    webdriver.ActionChains(driver).send_keys(Keys.ESCAPE).perform()

Let's set up our browser with Chrome and configure screenshot capabilities:

In [9]:
# --- Installation and PATH setup for Chromedriver in Colab (Revised) ---

# 1. Remove any pre-installed chromium and install google-chrome-stable (non-snap version)
#    This part is crucial to ensure a non-snap version of Chrome is used.
!apt-get purge chromium-browser chromium-chromedriver -y # Remove potentially problematic snap versions
!rm -rf /etc/apt/sources.list.d/google-chrome.list # Clean up old source list if it exists
!curl -sS -o - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" > /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install google-chrome-stable -y

# 2. Install webdriver-manager to automatically handle ChromeDriver installation
!pip install webdriver-manager

# 3. Import necessary components and ensure ChromeDriver path is set
import os
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Get the path to the compatible ChromeDriver executable
# This will download ChromeDriver if necessary and return its path
chromedriver_path = ChromeDriverManager().install()
print(f"ChromeDriver installed at: {chromedriver_path}")

# Ensure the directory of the downloaded chromedriver is in PATH
chromedriver_dir = os.path.dirname(chromedriver_path)
if chromedriver_dir not in os.environ['PATH']:
    os.environ['PATH'] += os.pathsep + chromedriver_dir
    print(f"Added {chromedriver_dir} to PATH.")


# Configure Chrome options
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless=new') # Use the newer headless mode
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--window-size=1920,1080') # Set a reasonable default window size for headless
chrome_options.add_argument('--remote-debugging-port=9222') # Often helps with headless stability
chrome_options.add_argument('--disable-extensions')
chrome_options.add_argument('--disable-setuid-sandbox') # Another sandbox-related option
chrome_options.add_argument('--disable-infobars') # Prevent infobars from appearing
chrome_options.add_argument('--log-level=3') # Suppress excessive logging from Chrome itself

# Explicitly set the path to the Chrome binary (for google-chrome-stable)
chrome_options.binary_location = '/usr/bin/google-chrome'

# Initialize the browser
driver = helium.start_chrome(headless=True, options=chrome_options) # headless=True is redundant with --headless=new but safe

# Set up screenshot callback
def save_screenshot(memory_step: ActionStep, agent: CodeAgent) -> None:
    sleep(1.0)  # Let JavaScript animations happen before taking the screenshot
    driver = helium.get_driver()
    current_step = memory_step.step_number
    if driver is not None:
        for previous_memory_step in agent.memory.steps:  # Remove previous screenshots for lean processing
            if isinstance(previous_memory_step, ActionStep) and previous_memory_step.step_number <= current_step - 2:
                previous_memory_step.observations_images = None
        png_bytes = driver.get_screenshot_as_png()
        image = Image.open(BytesIO(png_bytes))
        print(f"Captured a browser screenshot: {image.size} pixels")
        memory_step.observations_images = [image.copy()]  # Create a copy to ensure it persists

    # Update observations with current URL
    url_info = f"Current url: {driver.current_url}"
    memory_step.observations = (
        url_info if memory_step.observations is None else memory_step.observations + "\n" + url_info
    )

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following packages were automatically installed and are no longer required:
  apparmor libfuse3-3 snapd squashfs-tools systemd-hwe-hwdb udev
Use 'apt autoremove' to remove them.
The following packages will be REMOVED:
  chromium-browser* chromium-chromedriver*
0 upgraded, 0 newly installed, 2 to remove and 28 not upgraded.
After this operation, 124 kB disk space will be freed.
(Reading database ... 123389 files and directories currently installed.)
Removing chromium-chromedriver (2:1snap1-0ubuntu2) ...
Removing chromium-browser (2:1snap1-0ubuntu2) ...
Processing triggers for hicolor-icon-theme (0.17-2) ...
(Reading database ... 123368 files and directories currently installed.)
Purging configuration files for chromium-browser (2:1snap1-0ubuntu2) ...
OK
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [2,548 B]
Hit:2 http://security.ubuntu.com/ubuntu noble-security InRelease


In [14]:
from huggingface_hub import notebook_login

notebook_login()

The `notebook_login()` function can sometimes interfere with explicitly provided API keys. Since the token is already being passed directly to `InferenceClientModel` from Colab secrets, we can remove the `notebook_login()` call.

In [ ]:
# To avoid potential conflicts, the `notebook_login()` cell was removed.
# Your `hf_token` is already being passed directly to `InferenceClientModel`.

Now let's create our web automation agent:

In [30]:
from smolagents import InferenceClientModel
from google.colab import userdata

# Retrieve your Hugging Face API token from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Initialize the model
model_id = "qwen/qwen2-7b-instruct"  # Changed to a valid model ID suggested by the error
model = InferenceClientModel(model_id=model_id, api_key=hf_token)

# Create the agent
agent = CodeAgent(
    tools=[go_back, close_popups, search_item_ctrl_f],
    model=model,
    additional_authorized_imports=["helium"],
    step_callbacks=[save_screenshot],
    max_steps=20,
    verbosity_level=2,
)

The agent needs instructions on how to use Helium for web automation. Here are the instructions we'll provide:

In [19]:
helium_instructions = """
You can use helium to access websites. Don't bother about the helium driver, it's already managed.
We've already ran "from helium import *"
Then you can go to pages!
Code:
go_to('github.com/trending')
```<end_code>

You can directly click clickable elements by inputting the text that appears on them.
Code:
click("Top products")
```<end_code>

If it's a link:
Code:
click(Link("Top products"))
```<end_code>

If you try to interact with an element and it's not found, you'll get a LookupError.
In general stop your action after each button click to see what happens on your screenshot.
Never try to login in a page.

To scroll up or down, use scroll_down or scroll_up with as an argument the number of pixels to scroll from.
Code:
scroll_down(num_pixels=1200) # This will scroll one viewport down
```<end_code>

When you have pop-ups with a cross icon to close, don't try to click the close icon by finding its element or targeting an 'X' element (this most often fails).
Just use your built-in tool `close_popups` to close them:
Code:
close_popups()
```<end_code>

You can use .exists() to check for the existence of an element. For example:
Code:
if Text('Accept cookies?').exists():
    click('I accept')
```<end_code>
"""

Now we can run our agent with a task! Let's try finding information on Wikipedia:

In [32]:
search_request = """
Please navigate to https://en.wikipedia.org/wiki/Chicago and give me a sentence containing the word "1992" that mentions a construction accident.
"""

agent_output = agent.run(search_request + helium_instructions)
print("Final output:")
print(agent_output)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Please navigate to https://en.wikipedia.org/wiki/Chicago and give me a sentence containing the word "1992" that │
│ mentions a construction accident.                                                                               │
│                                                                                                                 │
│ You can use helium to access websites. Don't bother about the helium driver, it's already managed.              │
│ We've already ran "from helium import *"                                                                        │
│ Then you can go to pages!                                                                                       │
│ Code:                                                                                                           │
│ go_to('github.com/trending')                                                                                    │
│ ```<end_code>                                                                                                   │
│                                                                                                                 │
│ You can directly click clickable elements by inputting the text that appears on them.                           │
│ Code:                                                                                                           │
│ click("Top products")                                                                                           │
│ ```<end_code>                                                                                                   │
│                                                                                                                 │
│ If it's a link:                                                                                                 │
│ Code:                                                                                                           │
│ click(Link("Top products"))                                                                                     │
│ ```<end_code>                                                                                                   │
│                                                                                                                 │
│ If you try to interact with an element and it's not found, you'll get a LookupError.                            │
│ In general stop your action after each button click to see what happens on your screenshot.                     │
│ Never try to login in a page.                                                                                   │
│                                                                                                                 │
│ To scroll up or down, use scroll_down or scroll_up with as an argument the number of pixels to scroll from.     │
│ Code:                                                                                                           │
│ scroll_down(num_pixels=1200) # This will scroll one viewport down                                               │
│ ```<end_code>                                                                                                   │
│                                                                                                                 │
│ When you have pop-ups with a cross icon to close, don't try to click the close icon by finding its element or   │
│ targeting an 'X' element (this most often fails).                                                               │
│ Just use your built-in tool `close_popups` to close them:                                                       │
│ Code:                                                 

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
(Request ID: Root=1-6aa39806-7fc8783f24ad78f27bd742f8;eb8d96b1-fd51-4ac7-9c86-e86dcd6b75ed)

Bad request:
{'message': "The requested model 'qwen/qwen2-7b-instruct' does not exist.", 'type': 'invalid_request_error', 
'param': 'model', 'code': 'model_not_found'}

Captured a browser screenshot: (1920, 993) pixels


[Step 1: Duration 0.15 seconds]

AgentGenerationError: Error in generating model output:
(Request ID: Root=1-6aa39806-7fc8783f24ad78f27bd742f8;eb8d96b1-fd51-4ac7-9c86-e86dcd6b75ed)

Bad request:
{'message': "The requested model 'qwen/qwen2-7b-instruct' does not exist.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_found'}

You can run different tasks by modifying the request. For example, here's for me to know if I should work harder:

In [ ]:
github_request = """
I'm trying to find how hard I have to work to get a repo in github.com/trending.
Can you navigate to the profile for the top author of the top trending repo, and give me their total number of commits over the last year?
"""

agent_output = agent.run(github_request + helium_instructions)
print("Final output:")
print(agent_output)

The system is particularly effective for tasks like:
- Data extraction from websites
- Web research automation
- UI testing and verification
- Content monitoring